# Checkbook NYC API Proof-of-Concept: Live Data for One Agency

Small, self-contained demo: pull DOT spending data **live** from the
Checkbook NYC API instead of the pre-downloaded CSVs used everywhere else
in this project, to show the same analysis pipeline can reach any NYC
agency on demand, not just the ones already exported to `raw datasets/`.

**Endpoint confirmed by direct testing (not guessed):** `POST
https://www.checkbooknyc.com/api`, XML request body, XML response, no API
key. The schema below (nested `<name>`/`<type>`/`<value>` elements inside
each `<criteria>`, not XML attributes) was confirmed empirically -- an
attribute-based request (`<criteria name="agency" type="agency"
value="..."/>`) is accepted with `status=success` but **silently ignores
the filter** (returns unfiltered top records). The element-based form
below is what the API actually validates and applies; getting this wrong
fails silently, not loudly, which is worth knowing if you extend this.

**Scope, kept deliberately small:** one agency (DOT, `agency_code=841`),
one fiscal year (2023), `max_records=100` -- fast, and enough to prove the
pipeline works end to end without pulling a large page.

## Step 1: Build the XML Request

`agency_code` (not `agency`) and `type=value` (not a field-specific type
string) are the two details that aren't guessable from the field names
alone -- both confirmed by testing an invalid request and reading the
API's own validation error, which lists every valid Spending search-field
name (`fiscal_year, payee_code, payee_name, document_id, agency_code,
issue_date, conditional_category, department_code, check_amount,
expense_category, contract_id, capital_project_code, spending_category,
mwbe_category, industry`).

In [1]:
import requests
import xml.etree.ElementTree as ET

import pandas as pd

API_URL = "https://www.checkbooknyc.com/api"

# Confirmed by direct query against the live API (not looked up from a
# static table) -- see the markdown note below Step 4 for how.
DOT_AGENCY_CODE = "841"


def build_spending_request_xml(agency_code: str, fiscal_year: int, max_records: int,
                                records_from: int = 1,
                                columns=("agency", "fiscal_year", "department",
                                         "expense_category", "budget_code", "check_amount")) -> str:
    '''Build the XML request body for a Spending query filtered to one
    agency + one fiscal year. Each search criterion is a <criteria> element
    with nested <name>/<type>/<value> children -- NOT XML attributes (see
    the markdown note above for why that distinction matters here).
    '''
    criteria_xml = (
        f"<criteria><name>agency_code</name><type>value</type><value>{agency_code}</value></criteria>"
        f"<criteria><name>fiscal_year</name><type>value</type><value>{fiscal_year}</value></criteria>"
    )
    columns_xml = "".join(f"<column>{c}</column>" for c in columns)

    return (
        "<request>"
        "<type_of_data>Spending</type_of_data>"
        f"<records_from>{records_from}</records_from>"
        f"<max_records>{max_records}</max_records>"
        f"<search_criteria>{criteria_xml}</search_criteria>"
        f"<response_columns>{columns_xml}</response_columns>"
        "</request>"
    )


request_xml = build_spending_request_xml(agency_code=DOT_AGENCY_CODE, fiscal_year=2023, max_records=100)
print("Request body:\n")
print(request_xml)

Request body:

<request><type_of_data>Spending</type_of_data><records_from>1</records_from><max_records>100</max_records><search_criteria><criteria><name>agency_code</name><type>value</type><value>841</value></criteria><criteria><name>fiscal_year</name><type>value</type><value>2023</value></criteria></search_criteria><response_columns><column>agency</column><column>fiscal_year</column><column>department</column><column>expense_category</column><column>budget_code</column><column>check_amount</column></response_columns></request>


## Step 2: POST the Request

Wrapped in a `try/except` for network errors specifically -- if this
environment's outbound network is restricted (sandboxed CI, an
allowlisted proxy, etc.), this fails loudly with a clear message instead
of silently faking a response. **In this environment, outbound HTTPS to
`checkbooknyc.com` was verified working** before this notebook was
written (a `curl` test returned HTTP 200 in under a second) -- the
`try/except` here is for portability to other environments, not a hedge
against something already known to be broken.

In [2]:
try:
    response = requests.post(
        API_URL,
        data=request_xml.encode("utf-8"),
        headers={"Content-Type": "application/xml"},
        timeout=20,
    )
    print(f"HTTP status: {response.status_code}")
    network_ok = True
except requests.exceptions.RequestException as e:
    print("Network request failed -- this looks like an environment limitation")
    print("(sandboxed network, missing outbound allowlist entry for checkbooknyc.com, etc.),")
    print("not a bug in this notebook. Re-run this same cell from a normal terminal or")
    print("Google Colab, where outbound HTTPS isn't restricted.")
    print(f"\nError detail: {type(e).__name__}: {e}")
    network_ok = False
    response = None

HTTP status: 200


## Step 3: Parse the XML Response

Every response has a `<status><result>success|failure</result></status>`
element regardless of HTTP status code (HTTP 200 doesn't mean the *query*
succeeded) -- checked explicitly before trusting anything else in the
response. On failure, the API returns a `<messages><message>` block with a
real, specific description (confirmed while debugging Step 1's schema --
see the markdown note there); printed cleanly here rather than raising a
generic parse error.

In [3]:
records = []
api_error = None

if network_ok and response is not None and response.status_code == 200:
    root = ET.fromstring(response.text)

    result = root.findtext("./status/result")
    print(f"API status: {result}")

    if result == "success":
        record_count_matching = root.findtext("./result_records/record_count")
        print(f"Total records matching the filter (DOT, FY2023): {record_count_matching}")
        print(f"(This request asked for at most 100 of those -- see max_records in Step 1.)")

        for txn in root.findall("./result_records/spending_transactions/transaction"):
            records.append({child.tag: child.text for child in txn})

    else:
        messages = root.findall("./status/messages/message")
        for m in messages:
            code_ = m.findtext("code")
            desc = m.findtext("description")
            print(f"API error {code_}: {desc}")
        api_error = "; ".join(m.findtext("description") or "" for m in messages) or "Unknown API error"

elif network_ok and response is not None:
    print(f"Unexpected HTTP status {response.status_code} -- not a 200, and not a network exception either.")
    print("Response body (first 500 chars):")
    print(response.text[:500])

else:
    print("Skipped parsing -- no response to parse (see Step 2's network error above).")

API status: success
Total records matching the filter (DOT, FY2023): 77161
(This request asked for at most 100 of those -- see max_records in Step 1.)


## Step 4: Convert to a DataFrame

In [4]:
if records:
    df = pd.DataFrame(records)
    df["check_amount"] = pd.to_numeric(df["check_amount"], errors="coerce")

    print("Shape:", df.shape)
    print()
    df.head()
else:
    df = pd.DataFrame()
    print("No records parsed -- nothing to convert. See Steps 2-3 above for why "
          "(network failure, API error, or a genuinely empty result set).")

df.head() if not df.empty else df

Shape: (100, 6)



                         agency  ... fiscal_year
0  Department of Transportation  ...        2023
1  Department of Transportation  ...        2023
2  Department of Transportation  ...        2023
3  Department of Transportation  ...        2023
4  Department of Transportation  ...        2023

[5 rows x 6 columns]

**How `DOT_AGENCY_CODE = "841"` was confirmed, not assumed:** the
Spending domain's search fields require `agency_code` (a numeric NYC
agency code), not `agency` (a free-text name) -- discovered from the
API's own validation error when `agency` was tried first (Step 1's
note). `841` was then confirmed empirically by running this exact query
and checking that every returned `<agency>` value came back as
`"Department of Transportation"` -- not looked up from an external
table.

## Step 5: Summary -- Proof the Live Pull Worked

In [5]:
if not df.empty:
    total_spending = df["check_amount"].sum()
    n_records = len(df)

    print("=" * 60)
    print("LIVE PULL SUMMARY -- Checkbook NYC API")
    print("=" * 60)
    print(f"Agency:              Department of Transportation (agency_code={DOT_AGENCY_CODE})")
    print(f"Fiscal year:         2023")
    print(f"Records returned:    {n_records} (of {record_count_matching} total matching -- "
          f"capped by max_records)")
    print(f"Total check_amount:  ${total_spending:,.2f} (sum of the {n_records} returned rows only,")
    print(f"                     NOT the full FY2023 DOT total -- that would need paging")
    print(f"                     through all {record_count_matching} matching records)")
    print(f"Unique expense categories in this sample: {df['expense_category'].nunique()}")
else:
    print("No summary to show -- no records were returned (see Steps 2-4).")

LIVE PULL SUMMARY -- Checkbook NYC API
Agency:              Department of Transportation (agency_code=841)
Fiscal year:         2023
Records returned:    100 (of 77161 total matching -- capped by max_records)
Total check_amount:  $347,219,252.31 (sum of the 100 returned rows only,
                     NOT the full FY2023 DOT total -- that would need paging
                     through all 77161 matching records)
Unique expense categories in this sample: 12


## What This Demonstrates

This notebook pulls **live** DOT spending data directly from the
Checkbook NYC API -- the same underlying source the project's
pre-downloaded `raw datasets/DOT_*.csv` files came from originally
(confirmed: the live `budget_code` values returned here, e.g. `"4124
(TRAFFIC ENFORCEMENT CAMERA PROGRAM)"`, match the exact format already
seen in `DOT_Spending_Merged.csv`).

**Why this matters for the project's multi-agency goal:** every other
notebook in this project works from CSVs downloaded once for DOT (and, in
earlier work, TLC). This proof-of-concept shows the same
`agency_code` + `fiscal_year` request pattern generalizes to **any** NYC
agency the Checkbook NYC API covers, live, on demand -- swapping
`agency_code="841"` for any other agency's code is the only change needed
to pull a different agency's spending. That directly supports scaling
this project from "a couple of hand-picked agencies with downloaded CSVs"
toward the proposal's broader multi-agency budget recommendation-system
goal, without needing a fresh manual CSV export for every new agency.

**What this is NOT:** this is a proof-of-concept for the data-access
pattern only. It is **not wired into `dot_forecast_model.py`, the
classifier, or any other model in this project** -- those still run on
the corrected, cached CSVs, which is the right choice for anything that
needs to be backtested and reproduced exactly. Turning this live-pull
pattern into a real multi-agency pipeline would still need the same two
corrections already validated for DOT (agency-scope alignment,
expense/capital scope alignment) applied per-agency, plus pagination
handling for agencies whose full history exceeds the 20,000-records-per-call
limit -- neither is built here.